# Copy 200 reference-review CXR images

This notebook reads an existing frozen sample from a private JSON file; it does not resample.
Set `JAMIA_REFERENCE_SAMPLE` to a JSON array of 200 objects with `blinded_review_id`
(`RRV-0001` through `RRV-0200`), `image_path`, and `image_sha256`.
Derive this file from your authorized review crosswalk and frozen manifests.
Do not publish it. Images are hash-checked before copying and matching existing copies are skipped.
The download folder contains images named by blinded review ID. Handle these under dataset access terms.
Record your actual review procedure separately; exporting images does not attest to a review.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import os
import shutil
import tempfile

RERUN_DIR = Path(os.environ.get('JAMIA_RERUN_DIR', Path.cwd()))
OUTPUT_DIR = RERUN_DIR / 'sampled_images'
AUDIT_DIR = RERUN_DIR / 'results' / 'reference_review_image_export'
# Exact root relocation only; never select a file merely by matching its basename.
SOURCE_ROOT = Path(os.environ.get('JAMIA_DATA_ROOT', '/data/liangz2/openi'))
OLD_ROOT = Path(os.environ.get('JAMIA_OLD_DATA_ROOT', '/vf/users/liangz2/openi'))

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Copy destination:', OUTPUT_DIR)


In [ ]:
# Supply the frozen private sample; no patient paths or hashes are distributed.
SAMPLE_PATH = Path(os.environ.get("JAMIA_REFERENCE_SAMPLE", RERUN_DIR / "private_reference_sample.json"))
SAMPLE = json.loads(SAMPLE_PATH.read_text(encoding="utf-8"))


In [ ]:
assert len(SAMPLE) == 200
assert {row['blinded_review_id'] for row in SAMPLE} == {f'RRV-{i:04d}' for i in range(1, 201)}
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
audit_path = AUDIT_DIR / f'image_export_{run_id}.json'
audit = {'sample_seed': 20261028, 'expected_records': 200, 'complete': False,
         'sample_sha256': hashlib.sha256(json.dumps(SAMPLE, sort_keys=True).encode()).hexdigest(),
         'output_dir': str(OUTPUT_DIR), 'records': []}

def save_audit():
    audit_path.write_text(json.dumps(audit, indent=2), encoding='utf-8')

plans = []
for row in SAMPLE:
    original = Path(row['image_path'])
    candidates = []
    if original.is_relative_to(OLD_ROOT):
        candidates.append(SOURCE_ROOT / original.relative_to(OLD_ROOT))
    candidates.append(original)
    source = next((p for p in candidates if p.is_file()), None)
    target = OUTPUT_DIR / (row['blinded_review_id'] + original.suffix.lower())
    entry = dict(row, destination=str(target), status='pending')
    audit['records'].append(entry)
    if source is None:
        entry.update(status='missing_source', attempted_paths=[str(p) for p in candidates])
        continue
    entry['resolved_source'] = str(source)
    if sha256_path(source) != row['image_sha256']:
        entry['status'] = 'source_hash_mismatch'
        continue
    if target.is_symlink() or (target.exists() and
            (not target.is_file() or sha256_path(target) != row['image_sha256'])):
        entry['status'] = 'destination_conflict'
        continue
    entry['status'] = 'already_verified' if target.exists() else 'ready'
    plans.append((source, target, entry))

save_audit()
problems = [e for e in audit['records'] if e['status'] not in ('ready', 'already_verified')]
if problems:
    for entry in problems:
        print(entry['blinded_review_id'], entry['status'])
    raise RuntimeError(f'Preflight failed for {len(problems)} records; no images copied. See {audit_path}')
print('Preflight passed for all 200 images.', dict(Counter(e['status'] for e in audit['records'])))


In [ ]:
if problems or len(plans) != 200:
    raise RuntimeError('Run the preflight cell successfully before copying.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
try:
    for source, target, entry in plans:
        if target.exists():
            if target.is_symlink() or sha256_path(target) != entry['image_sha256']:
                raise RuntimeError(f'Existing destination changed: {target}')
            entry['status'] = 'already_verified'
            continue
        # Copy to a temporary file; publish exclusively only after verification.
        fd, temporary = tempfile.mkstemp(prefix='.copy-', dir=OUTPUT_DIR)
        os.close(fd)
        temporary = Path(temporary)
        try:
            shutil.copyfile(source, temporary)
            if sha256_path(temporary) != entry['image_sha256']:
                raise RuntimeError(f'Copied image fingerprint mismatch: {entry["blinded_review_id"]}')
            os.link(temporary, target)  # Fails safely if another process created target.
        finally:
            temporary.unlink(missing_ok=True)
        entry['status'] = 'copied'
    for _, target, entry in plans:
        if target.is_symlink() or sha256_path(target) != entry['image_sha256']:
            raise RuntimeError(f'Final verification failed: {target}')
    expected_names = {target.name for _, target, _ in plans}
    extras = [p.name for p in OUTPUT_DIR.iterdir() if p.name not in expected_names]
    if extras:
        raise RuntimeError(f'Unexpected files in download folder; inspect without deleting blindly: {extras}')
    audit['complete'] = True
except Exception as exc:
    audit['error'] = f'{type(exc).__name__}: {exc}'
    raise
finally:
    audit['finished_at_utc'] = datetime.now(timezone.utc).isoformat()
    save_audit()

print('SUCCESS: all 200 sampled images copied and SHA-256 verified.')
print('Download this folder:', OUTPUT_DIR)
print('Analyst-only audit (not needed for blinded review):', audit_path)
print('Copy summary:', dict(Counter(e['status'] for e in audit['records'])))
